# Text Analysis of Martial Arts-related Subreddits

The purpose of this project is to analyse the differences and similarities between different subreddits existing around a common topic, in this case martial arts. At first, I will collect Reddit data, analyse TF-IDF scores, and attempt to classify the subreddits using k-means and Naive Bayes algorithms. Then, I will introduce networks to visualise connections between threaded comments and users. Finally, I will make use of embeddings to further the analysis.

## I/ Collecting Reddit Data and Exploring TF-IDF Results

In [1]:
# Setup autoreload
%load_ext autoreload
%autoreload 2

# Create README.md 
# pip3 install nbconvert
# jupyter nbconvert --execute --to markdown MartialArtsRedditAnalysis.ipynb
# then rename to README.md

In [3]:
from reddit_helper import *

Let's first collect  50 posts from the r/MuayThai, r/bjj, r/MMA, and r/Boxing subreddits, and display some of their initial characteristics. This is done using the `RedditScraper` class created in the `reddit_helper.py` file. 

In [4]:
# Example subreddits
subreddits = ['MuayThai', 'bjj', 'MMA', 'Boxing']

# Analysis parameters
MAX_TERMS = 1000
MIN_DOC_FREQ = 2
LIMIT = 50
USERNAME = "matteolarrode"

# Initialize scraper
scraper = RedditScraper(
user_agent=f"SDS_textanalysis/1.0 (by /u/{USERNAME})"
)

# Analyze each subreddit independently
results = {}
submissions = {}

for subreddit in subreddits:
    print(f"\nAnalyzing r/{subreddit}...")

    # Collect posts
    submissions[subreddit] = scraper.get_subreddit_posts(subreddit, limit=LIMIT)

    # Analyze subreddit
    results[subreddit] = analyze_subreddit(
        submissions[subreddit],
        max_terms=MAX_TERMS,   # Maximum number of terms to keep
        min_doc_freq=MIN_DOC_FREQ   # Term must appear in at least min_doc_freq documents
    )

    # Print results for this subreddit
    print(f"\nVocabulary Statistics for r/{subreddit}:")
    print(f"Total words: {results[subreddit]['vocab_stats']['total_words']}")
    print(f"Unique words: {results[subreddit]['vocab_stats']['unique_words']}")
    print(f"Words appearing ≥{MIN_DOC_FREQ} times: {results[subreddit]['vocab_stats']['words_min_freq']}")
    print(f"Coverage by top {MAX_TERMS} words: {results[subreddit]['vocab_stats']['coverage_top_1000']:.2f}%")
    print(f"Matrix shape: {results[subreddit]['matrix_shape']}")
    print(f"Matrix sparsity: {results[subreddit]['matrix_sparsity']:.2f}%")

    print("\nTop 5 terms by TF-IDF score:")
    print(results[subreddit]['top_terms'][['term', 'score']].to_string())


Analyzing r/MuayThai...

Vocabulary Statistics for r/MuayThai:
Total words: 3233
Unique words: 1004
Words appearing ≥2 times: 426
Coverage by top 1000 words: 99.88%
Matrix shape: (50, 243)
Matrix sparsity: 93.83%

Top 5 terms by TF-IDF score:
         term     score
63      fight  0.058930
98         im  0.052751
132      muay  0.046444
217  training  0.046111
204      thai  0.042229

Analyzing r/bjj...

Vocabulary Statistics for r/bjj:
Total words: 5387
Unique words: 1460
Words appearing ≥2 times: 620
Coverage by top 1000 words: 91.46%
Matrix shape: (50, 405)
Matrix sparsity: 93.46%

Top 5 terms by TF-IDF score:
       term     score
36      bjj  0.065174
15   anyone  0.043297
164      im  0.042168
33   better  0.038398
123     get  0.037497

Analyzing r/MMA...

Vocabulary Statistics for r/MMA:
Total words: 1599
Unique words: 710
Words appearing ≥2 times: 247
Coverage by top 1000 words: 100.00%
Matrix shape: (50, 149)
Matrix sparsity: 94.50%

Top 5 terms by TF-IDF score:
      term  

Let's try to grasp a better understanding of the data structure of the `results` object.

In [5]:
# Data Exploration: 
submissions['MuayThai'][0] # Example post from the MuayThai subreddit

url_list = [post['url'] for post in submissions['MuayThai']]
url_df = pd.DataFrame(url_list, columns=['url'])
url_df['domain'] = url_df['url'].str.extract(r'(https?://[^/]+)')

url_df['domain'].value_counts().head(10)

domain
https://www.reddit.com     33
https://v.redd.it           8
https://i.redd.it           4
https://youtu.be            2
https://muay-ying.com       1
https://www.youtube.com     1
Name: count, dtype: int64

In [6]:
results['MuayThai'].keys()

dict_keys(['vocab_stats', 'freq_distribution', 'top_terms', 'vectorizer', 'matrix_shape', 'matrix_sparsity'])